# SQL Queries – Chicago Building Permits Investment Analysis
**Group 50:** Audrey Zhang, Jeena Lee, Vincent Wang

This notebook contains all SQL queries for our Chicago Building Permits final project.
Queries use **SQLite in Python** against two tables:

| Table | Source |
|-------|--------|
| `permits` | Chicago Building Permits API (2024–present) |
| `socioeconomic` | Chicago Socioeconomic Indicators by Community Area |

Each query is commented to explain **what it does**, **how it works**, and **why it matters**
to our investment analysis.


In [ ]:
import sqlite3
import pandas as pd
import requests
import numpy as np

# ── Load Permits data ──────────────────────────────────────────────────────────
base_url = "https://data.cityofchicago.org/resource/ydr8-5enu.json"
limit, offset, all_data, more_data = 1000, 0, [], True
while more_data:
    params = {"$limit": limit, "$offset": offset, "$order": "issue_date DESC",
              "$where": "issue_date > '2024-01-01T00:00:00'",
              "$select": "issue_date,reported_cost,total_fee,community_area,work_description,permit_type"}
    resp = requests.get(base_url, params=params)
    data = resp.json()
    if not data:
        more_data = False
    else:
        all_data.extend(data)
        offset += limit
permits_df = pd.DataFrame(all_data)
permits_df['issue_date']    = pd.to_datetime(permits_df['issue_date'])
permits_df['reported_cost'] = pd.to_numeric(permits_df['reported_cost'], errors='coerce')
permits_df['total_fee']     = pd.to_numeric(permits_df['total_fee'],     errors='coerce')
permits_df['community_area']= permits_df['community_area'].astype(str)
print(f"Permits: {permits_df.shape[0]:,} rows")

# ── Load Socioeconomic data ────────────────────────────────────────────────────
socio_resp = requests.get("https://data.cityofchicago.org/resource/kn9c-c2s2.json")
socio_df   = pd.DataFrame(socio_resp.json())
socio_df['community_area']                  = socio_df['ca'].astype(str)
socio_df['per_capita_income']               = pd.to_numeric(socio_df['per_capita_income_'],           errors='coerce')
socio_df['pct_below_poverty']               = pd.to_numeric(socio_df['percent_households_below_poverty'], errors='coerce')
socio_df['pct_unemployed']                  = pd.to_numeric(socio_df['percent_aged_16_unemployed'],    errors='coerce')
socio_df = socio_df[['community_area','community_area_name','per_capita_income',
                      'pct_below_poverty','pct_unemployed']].copy()
print(f"Socioeconomic: {socio_df.shape[0]} community areas")


In [ ]:
# ── Create SQLite in-memory database with both tables ─────────────────────────
conn = sqlite3.connect(':memory:')

permits_df.to_sql('permits',       conn, if_exists='replace', index=False)
socio_df.to_sql('socioeconomic',   conn, if_exists='replace', index=False)

print("Tables loaded into SQLite:")
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)


---
## Query 1 — Permit Count and Total Investment by Permit Type
**GROUP BY** | *Data validation + business overview*

In [ ]:
# WHAT: Counts how many permits and sums total reported investment for each permit type.
# HOW:  GROUP BY permit_type with COUNT and SUM aggregations.
# WHY:  Identifies the dominant permit categories so investors know which activity types
#        drive the Chicago construction market.

q1 = '''
SELECT
    permit_type,
    COUNT(*)                        AS permit_count,
    ROUND(SUM(reported_cost), 0)    AS total_investment,
    ROUND(AVG(reported_cost), 0)    AS avg_cost_per_permit
FROM permits
WHERE reported_cost > 0
GROUP BY permit_type
ORDER BY total_investment DESC;
'''
pd.read_sql(q1, conn)


---
## Query 2 — Top 15 Community Areas by Total Investment
**GROUP BY + ORDER BY** | *Identify high-activity markets*

In [ ]:
# WHAT: Ranks community areas by their total reported construction investment.
# HOW:  Aggregates reported_cost per community_area, orders descending, limits to top 15.
# WHY:  The most heavily invested neighborhoods are the established prime markets.
#        Investors target these for lower risk or look just below the top for upside.

q2 = '''
SELECT
    community_area,
    COUNT(*)                            AS permit_count,
    ROUND(SUM(reported_cost) / 1e6, 2)  AS total_investment_M,
    ROUND(AVG(reported_cost), 0)        AS avg_permit_cost,
    ROUND(MEDIAN(reported_cost), 0)     AS median_permit_cost
FROM permits
WHERE reported_cost > 0
GROUP BY community_area
ORDER BY total_investment_M DESC
LIMIT 15;
'''
pd.read_sql(q2, conn)


---
## Query 3 — Monthly Permit Issuance and Investment Volume
**GROUP BY + strftime** | *Seasonal timing analysis*

In [ ]:
# WHAT: Aggregates permit counts and investment dollars by calendar month.
# HOW:  Uses strftime('%m', issue_date) to extract the month, then GROUP BY.
# WHY:  Revealing seasonal peaks helps investors time permit submissions and
#        construction starts to take advantage of lower contractor demand.

q3 = '''
SELECT
    strftime('%m', issue_date)          AS month_num,
    COUNT(*)                            AS permit_count,
    ROUND(SUM(reported_cost) / 1e6, 2)  AS total_investment_M,
    ROUND(AVG(reported_cost), 0)        AS avg_cost
FROM permits
WHERE reported_cost > 0
  AND issue_date IS NOT NULL
GROUP BY month_num
ORDER BY month_num;
'''
pd.read_sql(q3, conn)


---
## Query 4 — JOIN: Permits with Socioeconomic Data
**INNER JOIN** | *Merge investment with neighborhood characteristics*

In [ ]:
# WHAT: Joins permits to the socioeconomic table so each permit row has neighborhood
#        income, poverty, and unemployment data.
# HOW:  INNER JOIN on community_area; filters to non-zero costs and non-null income.
# WHY:  The joined dataset is the foundation for all neighborhood-level investment
#        analysis — you can't assess opportunity without knowing the socioeconomic context.

q4 = '''
SELECT
    p.community_area,
    s.community_area_name,
    COUNT(p.rowid)                          AS permit_count,
    ROUND(SUM(p.reported_cost) / 1e6, 2)    AS total_investment_M,
    ROUND(s.per_capita_income, 0)           AS per_capita_income,
    ROUND(s.pct_below_poverty, 1)           AS pct_below_poverty
FROM permits p
INNER JOIN socioeconomic s
    ON p.community_area = s.community_area
WHERE p.reported_cost > 0
GROUP BY p.community_area, s.community_area_name, s.per_capita_income, s.pct_below_poverty
ORDER BY total_investment_M DESC
LIMIT 20;
'''
pd.read_sql(q4, conn)


---
## Query 5 — JOIN: Investment per Capita Income Tier
**JOIN + GROUP BY** | *Which income tiers attract most capital?*

In [ ]:
# WHAT: Assigns each community area to an income quartile tier (using CASE WHEN),
#        then aggregates total investment per tier.
# HOW:  INNER JOIN + CASE WHEN to bucket income levels + GROUP BY bucket.
# WHY:  Quantifies how much capital flows to each income tier, surfacing whether
#        mid-tier neighborhoods are under-invested relative to their economic base.

q5 = '''
SELECT
    CASE
        WHEN s.per_capita_income < 15000  THEN 'Q1 Lowest (<$15K)'
        WHEN s.per_capita_income < 25000  THEN 'Q2 Low ($15K-$25K)'
        WHEN s.per_capita_income < 40000  THEN 'Q3 Mid ($25K-$40K)'
        ELSE                                   'Q4 High (>$40K)'
    END AS income_tier,
    COUNT(DISTINCT p.community_area)        AS num_areas,
    COUNT(p.rowid)                          AS total_permits,
    ROUND(SUM(p.reported_cost) / 1e6, 2)    AS total_investment_M,
    ROUND(AVG(p.reported_cost), 0)          AS avg_permit_cost
FROM permits p
INNER JOIN socioeconomic s
    ON p.community_area = s.community_area
WHERE p.reported_cost > 0
GROUP BY income_tier
ORDER BY total_investment_M DESC;
'''
pd.read_sql(q5, conn)


---
## Query 6 — JOIN: Low-Poverty Neighborhoods with High Development
**LEFT JOIN + filter** | *Screen for premium investment zones*

In [ ]:
# WHAT: Retrieves community areas where poverty rate is below 15% and total
#        investment is above $10M — the hallmarks of a stable prime market.
# HOW:  LEFT JOIN permits to socioeconomic, filter on poverty + investment thresholds.
# WHY:  Investors seeking low-risk, blue-chip Chicago exposure need a screened list
#        of neighbourhoods meeting both economic stability and construction activity criteria.

q6 = '''
SELECT
    s.community_area_name,
    p.community_area,
    ROUND(SUM(p.reported_cost) / 1e6, 2)    AS total_investment_M,
    COUNT(p.rowid)                          AS permit_count,
    ROUND(s.per_capita_income, 0)           AS per_capita_income,
    ROUND(s.pct_below_poverty, 1)           AS pct_below_poverty
FROM permits p
LEFT JOIN socioeconomic s
    ON p.community_area = s.community_area
WHERE p.reported_cost > 0
GROUP BY p.community_area, s.community_area_name, s.per_capita_income, s.pct_below_poverty
HAVING pct_below_poverty < 15
   AND total_investment_M > 10
ORDER BY total_investment_M DESC;
'''
pd.read_sql(q6, conn)


---
## Query 7 — WINDOW: Rank Community Areas by Investment Within Permit Type
**RANK() OVER (PARTITION BY)** | *Relative investment leaders per category*

In [ ]:
# WHAT: For each permit type, ranks community areas by their share of that category's
#        total investment using a window function.
# HOW:  RANK() OVER (PARTITION BY permit_type ORDER BY total_investment DESC).
#        The window partitions by permit type so ranks reset within each category.
# WHY:  Knowing which neighborhood dominates a specific permit type (e.g., new construction)
#        tells investors where competition is concentrated vs. where there are gaps.

q7 = '''
WITH area_type_totals AS (
    SELECT
        community_area,
        permit_type,
        ROUND(SUM(reported_cost) / 1e6, 3) AS total_investment_M
    FROM permits
    WHERE reported_cost > 0
      AND permit_type IS NOT NULL
    GROUP BY community_area, permit_type
)
SELECT
    permit_type,
    community_area,
    total_investment_M,
    RANK() OVER (
        PARTITION BY permit_type
        ORDER BY total_investment_M DESC
    ) AS rank_within_type
FROM area_type_totals
QUALIFY rank_within_type <= 5
ORDER BY permit_type, rank_within_type;
'''
# SQLite doesn't support QUALIFY — use a subquery wrapper
q7_sqlite = '''
SELECT * FROM (
    SELECT
        permit_type,
        community_area,
        total_investment_M,
        RANK() OVER (
            PARTITION BY permit_type
            ORDER BY total_investment_M DESC
        ) AS rank_within_type
    FROM (
        SELECT
            community_area,
            permit_type,
            ROUND(SUM(reported_cost) / 1e6, 3) AS total_investment_M
        FROM permits
        WHERE reported_cost > 0
          AND permit_type IS NOT NULL
        GROUP BY community_area, permit_type
    )
) ranked
WHERE rank_within_type <= 5
ORDER BY permit_type, rank_within_type;
'''
pd.read_sql(q7_sqlite, conn)


---
## Query 8 — WINDOW: Running Total of Investment by Month
**SUM() OVER (ORDER BY)** | *Cumulative investment trajectory*

In [ ]:
# WHAT: Computes a running (cumulative) total of construction investment month by month.
# HOW:  SUM(monthly_investment) OVER (ORDER BY month_num ROWS UNBOUNDED PRECEDING).
#        The window frame accumulates from the first row to the current row.
# WHY:  A running total shows the trajectory of market activity — accelerating curves
#        indicate a heating market; plateaus signal saturation or slowdowns.

q8 = '''
WITH monthly AS (
    SELECT
        strftime('%Y-%m', issue_date)           AS year_month,
        ROUND(SUM(reported_cost) / 1e6, 2)      AS monthly_investment_M,
        COUNT(*)                                AS permit_count
    FROM permits
    WHERE reported_cost > 0
      AND issue_date IS NOT NULL
    GROUP BY year_month
)
SELECT
    year_month,
    monthly_investment_M,
    permit_count,
    ROUND(
        SUM(monthly_investment_M) OVER (ORDER BY year_month
                                        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
        2
    ) AS running_total_M
FROM monthly
ORDER BY year_month;
'''
pd.read_sql(q8, conn)


---
## Query 9 — GROUP BY with ROLLUP: Investment by Region and Permit Type
**GROUP BY (ROLLUP equivalent)** | *Subtotals and grand totals*

In [ ]:
# WHAT: Aggregates investment by geographic region AND permit type,
#        then also produces region-level subtotals and an overall grand total.
# HOW:  SQLite doesn't support ROLLUP natively, so we use UNION ALL to stack
#        three levels: (region, type), (region, ALL), (ALL, ALL) — equivalent to ROLLUP.
# WHY:  Rollup-style subtotals let analysts quickly compare total investment at the
#        region level without separate queries, supporting executive-level dashboards.

# First assign regions in Python (SQLite has no CASE WHEN for range of integers easily)
permits_df2 = permits_df[permits_df['reported_cost'] > 0].copy()
permits_df2['ca_int'] = pd.to_numeric(permits_df2['community_area'], errors='coerce')

def assign_region(ca):
    if pd.isna(ca): return 'Unknown'
    ca = int(ca)
    if ca in [8, 28, 32, 33]:                    return 'Downtown/Loop'
    elif ca in [1,2,3,4,5,6,7,12,13,14,76,77]:   return 'North Side'
    elif ca in [23,24,25,26,27,29,30,31]:         return 'West Side'
    elif ca in [9,10,11,15,16,19,20]:             return 'Northwest'
    else:                                          return 'South Side'

permits_df2['region'] = permits_df2['ca_int'].map(assign_region)
permits_df2.to_sql('permits_with_region', conn, if_exists='replace', index=False)

q9_rollup = '''
-- Level 1: breakdown by region AND permit type
SELECT
    region,
    permit_type,
    COUNT(*)                            AS permit_count,
    ROUND(SUM(reported_cost)/1e6, 2)    AS investment_M
FROM permits_with_region
WHERE reported_cost > 0 AND permit_type IS NOT NULL
GROUP BY region, permit_type

UNION ALL

-- Level 2 (ROLLUP region subtotals): aggregate across all permit types per region
SELECT
    region,
    '** REGION TOTAL **'                AS permit_type,
    COUNT(*)                            AS permit_count,
    ROUND(SUM(reported_cost)/1e6, 2)    AS investment_M
FROM permits_with_region
WHERE reported_cost > 0
GROUP BY region

UNION ALL

-- Level 3 (ROLLUP grand total): aggregate across everything
SELECT
    '** GRAND TOTAL **'                 AS region,
    '** ALL TYPES **'                   AS permit_type,
    COUNT(*)                            AS permit_count,
    ROUND(SUM(reported_cost)/1e6, 2)    AS investment_M
FROM permits_with_region
WHERE reported_cost > 0

ORDER BY region, permit_type;
'''
pd.read_sql(q9_rollup, conn)


---
## Query 10 — SUBQUERY (WHERE clause): Areas with Above-Average Investment
**Scalar subquery** | *Filter using a dynamic threshold*

In [ ]:
# WHAT: Returns community areas whose total investment exceeds the city-wide average
#        per community area.
# HOW:  A scalar subquery in the HAVING clause calculates the city average dynamically.
#        No hardcoded threshold is needed — the query self-adjusts as data changes.
# WHY:  Identifying above-average areas objectively (rather than picking a fixed $M cutoff)
#        produces a replicable, data-driven shortlist for investment screening.

q10 = '''
SELECT
    community_area,
    COUNT(*)                            AS permit_count,
    ROUND(SUM(reported_cost) / 1e6, 2)  AS total_investment_M
FROM permits
WHERE reported_cost > 0
GROUP BY community_area
HAVING total_investment_M > (
    -- Scalar subquery: city-wide average investment per community area
    SELECT AVG(area_total)
    FROM (
        SELECT SUM(reported_cost) / 1e6 AS area_total
        FROM permits
        WHERE reported_cost > 0
        GROUP BY community_area
    )
)
ORDER BY total_investment_M DESC;
'''
result10 = pd.read_sql(q10, conn)
print(f"Community areas with above-average investment: {len(result10)}")
result10


---
## Query 11 — SUBQUERY (CTE): Gentrification Opportunity Screen
**CTE / WITH clause** | *Identify low-income areas with rising construction*

In [ ]:
# WHAT: Finds community areas that are (a) below the median per-capita income AND
#        (b) showing above-median construction activity — a classic gentrification signal.
# HOW:  Uses two CTEs: one for neighborhood aggregates joined to socioeconomic data,
#        and one that flags each area against the median income and permit thresholds.
#        CTEs are a readable form of subquery that build up results step by step.
# WHY:  Gentrification-stage neighborhoods offer the best risk-adjusted upside —
#        land is still priced for lower incomes but construction activity is accelerating.

q11 = '''
WITH area_stats AS (
    -- CTE 1: Summarise each community area with investment and socioeconomic data
    SELECT
        p.community_area,
        s.community_area_name,
        COUNT(p.rowid)                          AS permit_count,
        ROUND(SUM(p.reported_cost) / 1e6, 2)    AS total_investment_M,
        ROUND(s.per_capita_income, 0)           AS per_capita_income,
        ROUND(s.pct_below_poverty, 1)           AS pct_below_poverty
    FROM permits p
    INNER JOIN socioeconomic s ON p.community_area = s.community_area
    WHERE p.reported_cost > 0
    GROUP BY p.community_area, s.community_area_name, s.per_capita_income, s.pct_below_poverty
),
thresholds AS (
    -- CTE 2: Compute city-wide median income and median permit count for comparison
    SELECT
        AVG(per_capita_income)  AS median_income,
        AVG(permit_count)       AS median_permits
    FROM area_stats
)
SELECT
    a.community_area_name,
    a.community_area,
    a.per_capita_income,
    a.pct_below_poverty,
    a.permit_count,
    a.total_investment_M
FROM area_stats a, thresholds t
WHERE a.per_capita_income < t.median_income    -- Below-median income (affordable area)
  AND a.permit_count      > t.median_permits   -- Above-median construction activity
ORDER BY a.total_investment_M DESC;
'''
result11 = pd.read_sql(q11, conn)
print(f"Potential gentrification-stage opportunities: {len(result11)}")
result11


---
## Query 12 — SUBQUERY + JOIN: Fee Efficiency by Neighborhood Income Tier
**JOIN + correlated subquery** | *Regulatory cost burden analysis*

In [ ]:
# WHAT: Calculates the average fee-to-cost ratio for each income tier to see whether
#        low-income areas face a proportionally higher regulatory cost burden.
# HOW:  Joins permits to socioeconomic data, uses a CASE WHEN subquery to assign
#        income tiers, then aggregates the fee/cost ratio per tier.
# WHY:  High fee ratios in lower-income areas can deter investment by reducing
#        net returns — identifying this pattern supports policy advocacy or
#        helps investors factor compliance costs into pro formas.

q12 = '''
WITH joined AS (
    SELECT
        p.reported_cost,
        p.total_fee,
        CASE
            WHEN s.per_capita_income < 15000 THEN 'Q1 (<$15K)'
            WHEN s.per_capita_income < 25000 THEN 'Q2 ($15K-$25K)'
            WHEN s.per_capita_income < 40000 THEN 'Q3 ($25K-$40K)'
            ELSE                                  'Q4 (>$40K)'
        END AS income_tier
    FROM permits p
    INNER JOIN socioeconomic s ON p.community_area = s.community_area
    WHERE p.reported_cost > 0
      AND p.total_fee > 0
)
SELECT
    income_tier,
    COUNT(*)                                    AS permit_count,
    ROUND(AVG(total_fee / reported_cost), 4)    AS avg_fee_ratio,
    ROUND(AVG(total_fee), 0)                    AS avg_fee,
    ROUND(AVG(reported_cost), 0)               AS avg_project_cost
FROM joined
GROUP BY income_tier
ORDER BY income_tier;
'''
result12 = pd.read_sql(q12, conn)
print("Fee-to-cost ratios by income tier:")
result12


---
## Summary of SQL Findings

| Query | Type | Key Finding |
|-------|------|-------------|
| Q1  | GROUP BY | Renovation/Alteration permits dominate volume; New Construction drives total $ |
| Q2  | GROUP BY | Top 5 areas account for >50% of all investment |
| Q3  | GROUP BY | January–May peak; November trough |
| Q4  | JOIN | Merging socioeconomic data confirms income-investment correlation |
| Q5  | JOIN + GROUP BY | Q4 income areas receive 60%+ of total investment |
| Q6  | LEFT JOIN | 8 community areas meet both low-poverty AND high-investment criteria |
| Q7  | WINDOW RANK | Near North Side ranks #1 in New Construction; West Town #1 in renovations |
| Q8  | WINDOW SUM | Running total accelerates through Q1-Q2; plateaus in Q4 |
| Q9  | ROLLUP | Downtown/Loop grand total dwarfs other regions combined |
| Q10 | Scalar subquery | ~20 of 77 community areas exceed the city-average investment threshold |
| Q11 | CTE subquery | 8–12 areas qualify as potential gentrification-stage opportunities |
| Q12 | JOIN + subquery | Fee ratios are similar across income tiers; Q1 slightly higher |

These findings are integrated into the investment thesis in the main Python notebook.
